### Data Collection


In [3]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd
import numpy as np

#loading the dataset
data = gutenberg.raw('shakespeare-hamlet.txt')
with open('shakespeare-hamlet.txt', 'w') as f:
    f.write(data)

[nltk_data] Downloading package gutenberg to /Users/apple/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


#### Data Preprocessing

In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential # type: ignore

#load the dataset
with open('shakespeare-hamlet.txt', 'r') as f:
    data = f.read().lower()
    
#tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index) + 1


In [6]:
total_words

4818

#### Input sequences

In [8]:
input_sequences = []
for line in data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [10]:
##Applying the padding
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

In [12]:
input_sequences=np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))


In [13]:
input_sequences 

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]], dtype=int32)

In [14]:
#create predictors and label
import tensorflow as tf
x,y=input_sequences[:,:-1],input_sequences[:,-1]

In [15]:
x


array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       ...,
       [   0,    0,    0, ...,  687,    4,   45],
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4]], dtype=int32)

In [16]:
y

array([ 687,    4,   45, ..., 1047,    4,  193], dtype=int32)

In [17]:
##convert y to categorical feature
y=tf.keras.utils.to_categorical(y, num_classes=total_words)

In [18]:
#Splitting the data into train and test data 
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

### LSTM Model Training

In [30]:
#Train the LSTM model

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


In [31]:
model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

#complile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_3 (Embedding)     (None, 13, 100)           481800    
                                                                 
 lstm_4 (LSTM)               (None, 13, 150)           150600    
                                                                 
 dropout_2 (Dropout)         (None, 13, 150)           0         
                                                                 
 lstm_5 (LSTM)               (None, 100)               100400    
                                                                 
 dense_2 (Dense)             (None, 4818)              486618    
                                                                 
Total params: 1219418 (4.65 MB)
Trainable params: 1219418 (4.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [32]:
## Training the model with train data
history = model.fit(x_train, y_train, epochs=50, verbose=1, validation_data=(x_test, y_test))

Epoch 1/50
644/644 [==============================] - 28s 37ms/step - loss: 6.9121 - accuracy: 0.0329 - val_loss: 6.6999 - val_accuracy: 0.0336
Epoch 2/50
644/644 [==============================] - 27s 42ms/step - loss: 6.4787 - accuracy: 0.0379 - val_loss: 6.7865 - val_accuracy: 0.0416
Epoch 3/50
644/644 [==============================] - 23s 35ms/step - loss: 6.3499 - accuracy: 0.0447 - val_loss: 6.8089 - val_accuracy: 0.0499
Epoch 4/50
644/644 [==============================] - 24s 38ms/step - loss: 6.2070 - accuracy: 0.0507 - val_loss: 6.8288 - val_accuracy: 0.0493
Epoch 5/50
644/644 [==============================] - 23s 36ms/step - loss: 6.0704 - accuracy: 0.0532 - val_loss: 6.8440 - val_accuracy: 0.0550
Epoch 6/50
644/644 [==============================] - 24s 37ms/step - loss: 5.9296 - accuracy: 0.0625 - val_loss: 6.9016 - val_accuracy: 0.0610
Epoch 7/50
644/644 [==============================] - 24s 37ms/step - loss: 5.7948 - accuracy: 0.0699 - val_loss: 6.9554 - val_accuracy:

In [43]:
#Function to generate the next words
def generate_text(model,tokenizer,text,max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len-1):]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=1)[0]
    for word,index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
            
    return None

In [51]:
input_text = "To be bad is better than "
print(f"Input Text: {input_text}")
max_sequence_len=model.input_shape[1]+1
next_word = generate_text(model,tokenizer,input_text,max_sequence_len)
print(f"Next word prediction: {next_word}")

Input Text: To be bad is better than 
Next word prediction: in


In [50]:
#save the model
model.save('lstm_text_generator.h5')
#save the tokenizer
import pickle
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

/Users/manishsabbani/anaconda3/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
